# 📘 智能体架构 11：元控制器

欢迎来到我们系列的第十一本笔记本。今天，我们将构建一个**元控制器**，这是一个编排专业子团队的管理型智能体架构。这一模式是创建强大的、多才多艺 AI 系统的基础。

与其构建一个试图做所有事情的单体智能体，元控制器充当一个智能调度器。它接收传入的请求，分析其性质，并将其路由到最合适的专业人员从可用智能体池中。这允许每个子智能体针对其特定任务进行高度优化，从而获得更好的性能和模块化。

我们将通过构建一个具有三个不同专业人员的系统来演示这一点：
1. **通才智能体：** 处理休闲对话和简单问题。
2. **研究智能体：** 配备搜索工具来回答有关最近事件或复杂主题的问题。
3. **编码智能体：** 专注于生成 Python 代码片段的专业人员。

元控制器将作为操作的"大脑"，检查每个用户查询并决定哪个智能体最适合响应。这创建了一个灵活且易于扩展的系统，可以通过创建新的专业智能体并教控制器了解它来简单地添加新功能。

### 定义
**元控制器**（或路由器）是多智能体系统中的管理型智能体，负责分析传入任务并将其分派给适当的专业子智能体或工作流程。它充当智能路由层，决定哪个工具或专家最适合当前的工作。

### 高层工作流程

1. **接收输入：** 系统接收用户请求。
2. **元控制器分析：** 元控制器智能体检查请求的意图、复杂性和内容。
3. **分派给专家：** 基于其分析，元控制器从预定义池中选择最佳专家智能体（例如，"研究员"、"程序员"、"聊天机器人"）。
4. **执行任务：** 所选专家智能体执行任务并生成结果。
5. **返回结果：** 专家的结果返回给用户。在更复杂的工作流程中，控制可能会返回元控制器进行进一步步骤或监控。

### 适用场景 / 应用
* **多服务 AI 平台：** 提供多样化服务（如文档分析、数据可视化、创意写作）的平台的单一入口点。
* **自适应个人助手：** 可以在不同模式或工具之间切换的助手，例如管理您的日历、搜索网络或控制智能家居设备。
* **企业工作流程：** 根据票据内容将客户支持票据路由到正确的部门（技术、计费、销售）。

### 优缺点
* **优点：**
    * **灵活性和模块化：** 非常容易添加新功能，只需添加新的专业智能体并更新控制器的路由逻辑。
    * **性能：** 允许高度优化的专家智能体，而不是可能在所有方面都平庸的万事通模型。
* **缺点：**
    * **控制器作为单点故障：** 整个系统的质量取决于控制器正确路由任务的能力。错误的路由决策导致次优或不正确的结果。
    * **潜在的延迟增加：** 与直接调用单个智能体相比，额外的路由步骤可能会增加少量延迟。

## 阶段 0：基础与环境设置

我们将安装库并设置我们的环境。我们需要 `langchain-tavily` 为我们研究智能体的搜索工具。

In [ ]:
# !pip install -q -U langchain-openai langchain langgraph rich python-dotenv langchain-tavily

In [ ]:
import os
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv

# Pydantic for data modeling
from pydantic import BaseModel, Field

# LangChain components
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate

# LangGraph components
from langgraph.graph import StateGraph, END
from typing_extensions import TypedDict

# For pretty printing
from rich.console import Console
from rich.markdown import Markdown

# --- API Key and Tracing Setup ---
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Agentic Architecture - Meta-Controller (OpenAI)"

required_vars = ["OPENAI_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
for var in required_vars:
    if var not in os.environ:
        print(f"Warning: Environment variable {var} not set.")

print("Environment variables loaded and tracing is set up.")

## 阶段 1：构建专家智能体

首先，我们将创建我们的专家智能体团队。每个智能体都是一个简单的链，具有特定的性格，对于研究员，还有一个工具。我们将它们包装在一个节点函数中，以便在我们的 LangGraph 中使用。

In [ ]:
console = Console()
model = os.environ.get("OPENAI_API_MODEL", "gpt-4o")
base_url = os.environ.get("OPENAI_API_BASE_URL", "https://api.openai.com/v1")
llm = ChatOpenAI(model=model, base_url=base_url, temperature=0)
search_tool = TavilySearch(max_results=3)

# Define the state for the overall graph
class MetaAgentState(TypedDict):
    user_request: str
    next_agent_to_call: Optional[str]
    generation: str

# A helper factory function to create specialist agent nodes
def create_specialist_node(persona: str, tools: list = None):
    """Factory to create a specialist agent node."""
    system_prompt = f"You are a specialist agent with the following persona: {persona}. Respond directly and concisely to the user's request based on your role."
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{user_request}")
    ])
    
    if tools:
        chain = prompt | llm.bind_tools(tools)
    else:
        chain = prompt | llm
        
    def specialist_node(state: MetaAgentState) -> Dict[str, Any]:
        result = chain.invoke({"user_request": state['user_request']})
        return {"generation": result.content}
    
    return specialist_node

# 1. Generalist Agent Node
generalist_node = create_specialist_node(
    "You are a friendly and helpful generalist AI assistant. You handle casual conversation and simple questions."
)

# 2. Research Agent Node
research_agent_node = create_specialist_node(
    "You are an expert researcher. You must use your search tool to find information to answer the user's question.",
    tools=[search_tool]
)

# 3. Coding Agent Node
coding_agent_node = create_specialist_node(
    "You are an expert Python programmer. Your task is to write clean, efficient Python code based on the user's request. Provide only the code, wrapped in markdown code blocks, with minimal explanation."
)

print("Specialist agents defined successfully.")

## 阶段 2：构建元控制器

这是我们系统的大脑。元控制器是一个由 LLM 驱动的节点，其唯一工作是决定将哪个专家路由请求到。其提示的质量对于系统性能至关重要。

In [ ]:
# Pydantic model for the controller's routing decision
class ControllerDecision(BaseModel):
    next_agent: str = Field(description="The name of the specialist agent to call next. Must be one of ['Generalist', 'Researcher', 'Coder'].")
    reasoning: str = Field(description="A brief reason for choosing the next agent.")

def meta_controller_node(state: MetaAgentState) -> Dict[str, Any]:
    """The central controller that decides which specialist to call."""
    console.print("--- 🧠 元控制器分析请求 ---")
    
    # Define the specialists and their descriptions for the controller
    specialists = {
        "Generalist": "Handles casual conversation, greetings, and simple questions.",
        "Researcher": "Answers questions about recent events, complex topics, or anything requiring up-to-date information from the web.",
        "Coder": "Writes Python code based on a user's specification."
    }
    
    specialist_descriptions = "\n".join([f"- {name}: {desc}" for name, desc in specialists.items()])
    
    prompt = ChatPromptTemplate.from_template(
        f"""你是多智能体 AI 系统的元控制器。你的工作是分析用户的请求并将其路由到最合适的专业智能体。

以下是可用的专家：
{specialist_descriptions}

分析以下用户请求并选择处理它的最佳专家。以要求的格式提供你的决定。

用户请求："{{user_request}}""""
    )
    
    controller_llm = llm.with_structured_output(ControllerDecision)
    chain = prompt | controller_llm
    
    decision = chain.invoke({"user_request": state['user_request']})
    console.print(f"[yellow]路由决策：[/yellow] 发送到 [bold]{decision.next_agent}[/bold]。[italic]原因：{decision.reasoning}[/italic]")
    
    return {"next_agent_to_call": decision.next_agent}

print("元控制器节点定义成功。")

## 阶段 3：组装和运行图

现在我们将使用 LangGraph 将所有内容连接在一起。图将从元控制器开始，然后基于其决定，条件边将状态路由到正确的专家节点。专家运行后，图将结束。

In [5]:
# Build the graph
workflow = StateGraph(MetaAgentState)

# Add nodes for the controller and each specialist
workflow.add_node("meta_controller", meta_controller_node)
workflow.add_node("Generalist", generalist_node)
workflow.add_node("Researcher", research_agent_node)
workflow.add_node("Coder", coding_agent_node)

# Set the entry point
workflow.set_entry_point("meta_controller")

# Define the conditional routing logic
def route_to_specialist(state: MetaAgentState) -> str:
    """Reads the controller's decision and returns the name of the node to route to."""
    return state["next_agent_to_call"]

workflow.add_conditional_edges(
    "meta_controller",
    route_to_specialist,
    {
        "Generalist": "Generalist",
        "Researcher": "Researcher",
        "Coder": "Coder"
    }
)

# After any specialist runs, the process ends
workflow.add_edge("Generalist", END)
workflow.add_edge("Researcher", END)
workflow.add_edge("Coder", END)

meta_agent = workflow.compile()
print("Meta-Controller agent graph compiled successfully.")

Meta-Controller agent graph compiled successfully.


## 阶段 4：演示

让我们用各种提示测试我们的元控制器，看看它是否正确地将它们分派给正确的专家。

In [6]:
def run_agent(query: str):
    result = meta_agent.invoke({"user_request": query})
    console.print("\n[bold]Final Response:[/bold]")
    console.print(Markdown(result['generation']))

# Test 1: Should be routed to the Generalist
console.print("--- 💬 Test 1: General Conversation ---")
run_agent("Hello, how are you today?")

# Test 2: Should be routed to the Researcher
console.print("\n--- 🔬 Test 2: Research Question ---")
run_agent("What were NVIDIA's latest financial results?")

# Test 3: Should be routed to the Coder
console.print("\n--- 💻 Test 3: Coding Request ---")
run_agent("Can you write me a python function to calculate the nth fibonacci number?")

--- 💬 Test 1: General Conversation ---


--- 🧠 Meta-Controller Analyzing Request ---
Routing decision: Send to Generalist. Reason: The user's request is a simple greeting, which falls under the category of casual conversation handled by the Generalist agent.



Final Response:


Hello there! How can I help you today?


--- 🔬 Test 2: Research Question ---


...

NVIDIA's latest financial results, for the quarter ending in April 2024, were exceptionally strong. They reported revenue of $26.04 billion, a significant increase year-over-year, driven largely by their Data Center revenue which hit a record $22.6 billion. Their GAAP earnings per diluted share were $5.98.


--- 💻 Test 3: Coding Request ---


--- 🧠 Meta-Controller Analyzing Request ---
Routing decision: Send to Coder. Reason: The user is explicitly asking for a Python function, which is a coding task. The Coder agent is the specialist for this.



Final Response:


```python
def fibonacci(n):
    """Calculates the nth Fibonacci number."""
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b
```

## 结论

在本笔记本中，我们成功实现了**元控制器**架构。我们的测试清楚地展示了其主要功能：充当智能和动态路由器。

1. 简单的问候被正确识别并发送给**通才**。
2. 关于最近财务新闻的查询被分派给**研究员**，它使用其搜索工具获取最新信息。
3. 代码片段请求被路由到**编码员**，它提供了格式良好且正确的函数。

这一模式对于构建可扩展和可维护的 AI 系统特别强大。通过分离关注点，每个专家都可以独立改进而不影响其他。通过添加新的、能力更强的专业人员并使元控制器了解它们，可以增强系统的整体智能。虽然控制器本身代表了潜在的瓶颈，但其作为灵活编排器的角色是高级智能体设计的基石。